In [ ]:
# Setup and imports
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import savgol_filter

from app.database import CourseDatabase
from app.course import Course

## 1. Load Raw GPS Data

Choose your data source: database sample course, CSV file, or Strava API

In [ ]:
# Option 1: Load from database
db = CourseDatabase()
courses = db.conn.execute(
    "SELECT course_id, course_name, total_distance_km, total_elevation_gain_m, course_type FROM courses"
).fetchall()

print("Available courses:")
for i, c in enumerate(courses):
    print(f"  {i}: {c['course_name']} ({c['course_type']}) - {c['total_distance_km']:.1f}km, {c['total_elevation_gain_m']:.0f}m")

In [ ]:
# Select a course (change index as needed)
COURSE_INDEX = 0  # <-- CHANGE THIS to select different course

selected_course = courses[COURSE_INDEX]
course_data = db.get_course(selected_course['course_id'])

# Get raw GPS data (before processing)
raw_df = pd.DataFrame(course_data['raw_gps_data'])
print(f"Loaded: {selected_course['course_name']}")
print(f"Points: {len(raw_df)}")
print(f"Columns: {list(raw_df.columns)}")

## 2. Raw Data Exploration

In [ ]:
# View raw altitude profile
fig = px.line(raw_df, y='altitude', title='Raw Altitude Profile')
fig.update_layout(xaxis_title='Point Index', yaxis_title='Altitude (m)')
fig.show()

In [ ]:
# Calculate raw gradient (without smoothing)
raw_df['delta_lat'] = raw_df['lat'].diff()
raw_df['delta_lon'] = raw_df['lon'].diff()
raw_df['distance'] = ((raw_df['delta_lat']**2 + raw_df['delta_lon']**2)**0.5) * 111139
raw_df['delta_altitude'] = raw_df['altitude'].diff()
raw_df['raw_gradient'] = (raw_df['delta_altitude'] / raw_df['distance']) * 100
raw_df['raw_gradient'] = raw_df['raw_gradient'].replace([np.inf, -np.inf], np.nan)

print(f"Raw gradient range: {raw_df['raw_gradient'].min():.1f}% to {raw_df['raw_gradient'].max():.1f}%")
print(f"Points with extreme gradient (>30%): {(raw_df['raw_gradient'].abs() > 30).sum()}")

In [ ]:
# Visualize raw gradient - notice the noise!
fig = px.line(raw_df, y='raw_gradient', title='Raw Gradient (Unsmoothed) - Notice the GPS Noise')
fig.update_layout(xaxis_title='Point Index', yaxis_title='Gradient (%)')
fig.add_hline(y=30, line_dash='dash', line_color='red', annotation_text='30% threshold')
fig.add_hline(y=-30, line_dash='dash', line_color='red')
fig.show()

## 3. Experiment with Smoothing Parameters

The Savitzky-Golay filter smooths altitude while preserving peaks and valleys. Key parameters:
- **window_length**: Number of points to include (must be odd)
- **polyorder**: Polynomial order for fitting (2 works well for altitude)

In [ ]:
# Smoothing parameters - EXPERIMENT WITH THESE
WINDOW_LENGTH = 21  # Must be odd, try: 5, 11, 21, 31, 51
POLYORDER = 2       # Polynomial order, typically 2 or 3
MAX_GRADIENT = 30.0 # Maximum gradient in percent

# Apply Savitzky-Golay smoothing
if WINDOW_LENGTH >= 5 and len(raw_df) >= WINDOW_LENGTH:
    altitude_smoothed = savgol_filter(
        raw_df['altitude'].ffill().bfill(),
        window_length=WINDOW_LENGTH,
        polyorder=POLYORDER
    )
else:
    altitude_smoothed = raw_df['altitude'].values

raw_df['altitude_smoothed'] = altitude_smoothed

In [ ]:
# Compare raw vs smoothed altitude
fig = go.Figure()
fig.add_trace(go.Scatter(y=raw_df['altitude'], name='Raw Altitude', line=dict(color='lightgray')))
fig.add_trace(go.Scatter(y=raw_df['altitude_smoothed'], name=f'Smoothed (window={WINDOW_LENGTH})', line=dict(color='blue')))
fig.update_layout(
    title=f'Altitude Smoothing Comparison (window={WINDOW_LENGTH}, polyorder={POLYORDER})',
    xaxis_title='Point Index',
    yaxis_title='Altitude (m)'
)
fig.show()

In [ ]:
# Calculate gradient from smoothed altitude
raw_df['delta_altitude_smooth'] = pd.Series(raw_df['altitude_smoothed']).diff()
raw_df['smooth_gradient'] = (raw_df['delta_altitude_smooth'] / raw_df['distance']) * 100
raw_df['smooth_gradient'] = raw_df['smooth_gradient'].replace([np.inf, -np.inf], np.nan)

# Apply clamping
raw_df['clamped_gradient'] = raw_df['smooth_gradient'].clip(lower=-MAX_GRADIENT, upper=MAX_GRADIENT)

print(f"Smoothed gradient range: {raw_df['smooth_gradient'].min():.1f}% to {raw_df['smooth_gradient'].max():.1f}%")
print(f"After clamping (±{MAX_GRADIENT}%): {raw_df['clamped_gradient'].min():.1f}% to {raw_df['clamped_gradient'].max():.1f}%")

In [ ]:
# Compare raw vs smoothed gradient
fig = make_subplots(rows=2, cols=1, subplot_titles=['Raw Gradient', 'Smoothed + Clamped Gradient'])

fig.add_trace(
    go.Scatter(y=raw_df['raw_gradient'], name='Raw', line=dict(color='red', width=0.5)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(y=raw_df['clamped_gradient'], name='Smoothed+Clamped', line=dict(color='blue')),
    row=2, col=1
)

fig.update_layout(height=600, title_text=f'Gradient Comparison (window={WINDOW_LENGTH}, clamp=±{MAX_GRADIENT}%)')
fig.show()

## 4. Adaptive Window Size

The production code uses an adaptive window that targets ~50m of horizontal distance. This adapts to different GPS sampling densities.

In [ ]:
# Calculate adaptive window size (as in production code)
avg_point_spacing = raw_df['distance'].median()
print(f"Median point spacing: {avg_point_spacing:.1f}m")

TARGET_DISTANCE = 50  # meters - EXPERIMENT WITH THIS

target_points = max(5, min(51, int(TARGET_DISTANCE / avg_point_spacing)))
adaptive_window = target_points if target_points % 2 == 1 else target_points + 1

print(f"Target distance for smoothing: {TARGET_DISTANCE}m")
print(f"Calculated adaptive window: {adaptive_window} points")
print(f"Actual smoothing distance: {adaptive_window * avg_point_spacing:.1f}m")

In [ ]:
# Apply adaptive smoothing
altitude_adaptive = savgol_filter(
    raw_df['altitude'].ffill().bfill(),
    window_length=adaptive_window,
    polyorder=2
)

raw_df['altitude_adaptive'] = altitude_adaptive
raw_df['delta_altitude_adaptive'] = pd.Series(altitude_adaptive).diff()
raw_df['adaptive_gradient'] = (raw_df['delta_altitude_adaptive'] / raw_df['distance']) * 100
raw_df['adaptive_gradient'] = raw_df['adaptive_gradient'].replace([np.inf, -np.inf], np.nan)
raw_df['adaptive_gradient_clamped'] = raw_df['adaptive_gradient'].clip(lower=-30, upper=30)

print(f"Adaptive gradient range: {raw_df['adaptive_gradient_clamped'].min():.1f}% to {raw_df['adaptive_gradient_clamped'].max():.1f}%")

## 5. Gradient Distribution Analysis

In [ ]:
# Histogram comparison
fig = go.Figure()
fig.add_trace(go.Histogram(x=raw_df['raw_gradient'].dropna(), name='Raw', opacity=0.5, nbinsx=50))
fig.add_trace(go.Histogram(x=raw_df['adaptive_gradient_clamped'].dropna(), name='Smoothed+Clamped', opacity=0.5, nbinsx=50))
fig.update_layout(
    title='Gradient Distribution: Raw vs Smoothed',
    xaxis_title='Gradient (%)',
    yaxis_title='Count',
    barmode='overlay'
)
fig.show()

In [ ]:
# Summary statistics
print("Gradient Statistics Comparison:")
print("="*50)
print(f"{'Metric':<20} {'Raw':>12} {'Smoothed':>12}")
print("-"*50)
print(f"{'Min':<20} {raw_df['raw_gradient'].min():>12.1f}% {raw_df['adaptive_gradient_clamped'].min():>12.1f}%")
print(f"{'Max':<20} {raw_df['raw_gradient'].max():>12.1f}% {raw_df['adaptive_gradient_clamped'].max():>12.1f}%")
print(f"{'Mean':<20} {raw_df['raw_gradient'].mean():>12.2f}% {raw_df['adaptive_gradient_clamped'].mean():>12.2f}%")
print(f"{'Std Dev':<20} {raw_df['raw_gradient'].std():>12.2f}% {raw_df['adaptive_gradient_clamped'].std():>12.2f}%")
print(f"{'Points > 30%':<20} {(raw_df['raw_gradient'].abs() > 30).sum():>12} {(raw_df['adaptive_gradient_clamped'].abs() > 30).sum():>12}")

## 6. Process with Production Course Class

Finally, let's verify the production `Course` class produces the same results.

In [ ]:
# Create Course object (applies production smoothing)
course = Course(pd.DataFrame(course_data['raw_gps_data']))

print(f"Production Course object:")
print(f"  Points: {len(course.df)}")
print(f"  Gradient range: {course.df['gradient'].min():.1f}% to {course.df['gradient'].max():.1f}%")
print(f"  Total distance: {course.df['cumulative_distance'].iloc[-1]/1000:.2f} km")
print(f"  Total elevation gain: {course.df['height_gain'].sum():.0f} m")